In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.formula.api as smf

In [2]:
df = pd.read_csv('../data/cleaned_maternal_deaths.csv')
df.head()

,state,state_code,race,race_code,deaths,population,crude_rate,ci_lower,ci_upper
0,Alabama,1.0,Black or African American,2054-5,65.0,9384181.0,0.7,0.5,0.9
1,Alabama,1.0,White,2106-3,60.0,24197348.0,0.2,0.2,0.3
2,Alaska,2.0,White,2106-3,10.0,3318369.0,0.3,0.1,0.6
3,Arizona,4.0,American Indian or Alaska Native,1002-5,18.0,2694672.0,0.7,0.4,1.1
4,Arizona,4.0,Black or African American,2054-5,18.0,2805255.0,0.6,0.4,1.0


In [4]:
df.shape

(86, 9)

In [6]:
ratio_data = df[df['race'].isin(['Black or African American', 'White'])].pivot(index='state', columns='race', values='crude_rate').dropna()
ratio_data.columns = ['Black', 'White']
ratio_data['ratio'] = ratio_data['Black'] / ratio_data['White']
ratio_data = ratio_data.sort_values('ratio', ascending=True)  
ratio_data.head()

,Black,White,ratio
state,,,
Maryland,0.3,0.2,1.500000
Texas,0.6,0.3,2.000000
Kentucky,0.6,0.3,2.000000
Indiana,0.7,0.3,2.333333
Nevada,0.5,0.2,2.500000


In [10]:
overall_rate = df.groupby('state').agg(
    total_deaths=('deaths', 'sum'),
    total_population=('population', 'sum')
).reset_index()
overall_rate['overall_rate'] = overall_rate['total_deaths'] / overall_rate['total_population'] * 100000
overall_rate = overall_rate.set_index('state')['overall_rate']
overall_rate.head()

state
Alabama       0.372228
Alaska        0.301353
Arizona       0.293225
Arkansas      0.408320
California    0.118231
Name: overall_rate, dtype: float64

In [11]:
n_total = len(ratio_data)
n_higher = (ratio_data['ratio'] > 1).sum()
n_lower = (ratio_data['ratio'] < 1).sum()
n_equal = (ratio_data['ratio'] == 1).sum()

print(f"Total states with data: {n_total}")
print(f"States where Black rate > White rate: {n_higher} ({n_higher/n_total*100:.1f}%)")
print(f"States where Black rate < White rate: {n_lower} ({n_lower/n_total*100:.1f}%)")
print(f"States where rates are equal: {n_equal}")

Total states with data: 31
States where Black rate > White rate: 31 (100.0%)
States where Black rate < White rate: 0 (0.0%)
States where rates are equal: 0


In [12]:
ratio_data['difference'] = ratio_data['Black'] - ratio_data['White']

print("--- Ratio summary ---")
print(ratio_data['ratio'].describe())

print("\n--- Difference summary ---")
print(ratio_data['difference'].describe())

--- Ratio summary ---
count    31.000000
mean      3.489247
std       1.119636
min       1.500000
25%       3.000000
50%       3.000000
75%       4.000000
max       6.500000
Name: ratio, dtype: float64

--- Difference summary ---
count    31.000000
mean      0.429032
std       0.186536
min       0.100000
25%       0.300000
50%       0.400000
75%       0.500000
max       1.100000
Name: difference, dtype: float64


In [15]:
print("Highest Black mortality rate:")
print(ratio_data['Black'].nlargest(5))

print("\nHighest White mortality rate:")
print(ratio_data['White'].nlargest(5))

print("\nHighest disparity ratio (Black/White):")
print(ratio_data['ratio'].nlargest(5))

print("\nHighest overall mortality rate:")
print(overall_rate.nlargest(5))

Highest Black mortality rate:
state
Kansas       1.3
Tennessee    1.0
Arkansas     1.0
Oklahoma     0.9
Louisiana    0.8
Name: Black, dtype: float64

Highest White mortality rate:
state
Texas        0.3
Kentucky     0.3
Indiana      0.3
Louisiana    0.3
Oklahoma     0.3
Name: White, dtype: float64

Highest disparity ratio (Black/White):
state
Kansas          6.5
Florida         6.0
Michigan        5.0
Pennsylvania    5.0
Wisconsin       5.0
Name: ratio, dtype: float64

Highest overall mortality rate:
state
District of Columbia    0.639269
Louisiana               0.460508
Tennessee               0.440870
Mississippi             0.424615
Arkansas                0.408320
Name: overall_rate, dtype: float64


In [16]:
mean_ratio = ratio_data['ratio'].mean()
std_ratio = ratio_data['ratio'].std()
outliers = ratio_data[ratio_data['ratio'] > mean_ratio + 1.5*std_ratio]
print("States with unusually high disparity (>1.5 SD above mean):")
print(outliers[['Black', 'White', 'ratio']])

States with unusually high disparity (>1.5 SD above mean):
         Black  White  ratio
state                       
Florida    0.6    0.1    6.0
Kansas     1.3    0.2    6.5


In [18]:
df_reg = df.copy()
df_reg['race_clean'] = df_reg['race'].str.replace(' ', '_').str.replace('/', '_')
df_reg['state_clean'] = df_reg['state'].str.replace(' ', '_')

model = smf.ols('crude_rate ~ C(race_clean, Treatment(reference="White")) + C(state_clean)', data=df_reg).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             crude_rate   R-squared:                       0.887
Model:                            OLS   Adj. R-squared:                  0.726
Method:                 Least Squares   F-statistic:                     5.512
Date:                Sun, 30 Aug 2026   Prob (F-statistic):           4.03e-07
Time:                        10:14:40   Log-Likelihood:                 89.838
No. Observations:                  86   AIC:                            -77.68
Df Residuals:                      35   BIC:                             47.50
Df Model:                          50                                         
Covariance Type:            nonrobust                                         
                                                                                      coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------

In [ ]:
results_df = pd.DataFrame({
    'coefficient': model.params,
    'std_err': model.bse,
    't_value': model.tvalues,
    'p_value': model.pvalues,
    'ci_lower': model.conf_int()[0],
    'ci_upper': model.conf_int()[1]
})
results_df.to_csv('regression_results.csv')
results_df.head(10)